# GAN 출력 방식 점검 — 추가 학습 없음

기존 베드 포함 GAN과 마스킹 GAN의 저장된 가중치를 읽어 같은 입력 16개를 넣습니다. 학습할 때처럼 배치 정규화 통계를 계산한 경우와 저장된 통계를 사용하는 평가 모드를 비교해, 회색 출력이 두 방식의 차이와 관련 있는지 살펴봅니다. 원인 확인용 추론이며 가중치를 수정하거나 모델을 학습하지 않습니다. 두 모드는 각각 동일한 체크포인트에서 별도의 모델을 로드하고, 학습 모드에서 바뀌는 임시 통계는 버립니다. 결과를 Drive에 저장한 뒤 G4 런타임을 반납합니다.

In [ ]:
from google.colab import runtime, drive
import sys, os, json, traceback, zipfile, hashlib, csv
from pathlib import Path

mounted=False
run_root=None
try:
    drive.mount('/content/drive'); mounted=True
    import torch
    from torchvision.utils import save_image
    from PIL import Image, ImageDraw, ImageOps

    drive_root=Path('/content/drive/MyDrive/산공종설')
    code_zip=drive_root/'gan_masked_pilot_code_20261101.zip'
    masked=drive_root/'results/gan_masked_20261101_run01/gan/checkpoint.pt'
    bed=drive_root/'results/gan_bed_20261101_run01/gan/checkpoint.pt'
    run_root=drive_root/'results/gan_batchnorm_diagnostic_20261101_run01'
    if run_root.exists() and any(run_root.iterdir()):
        previous=run_root; run_root=None
        raise FileExistsError(f'기존 결과 보호: {previous}. 다른 출력 이름으로 실행하세요.')
    run_root.mkdir(parents=True,exist_ok=True)
    if not torch.cuda.is_available(): raise RuntimeError('G4 GPU가 필요합니다.')
    gpu=torch.cuda.get_device_name(0)
    print('사용 GPU:',gpu,flush=True)
    if 'RTX PRO 6000' not in gpu.upper(): raise RuntimeError(f'G4가 아닌 GPU: {gpu}')
    for p in [code_zip,masked,bed]:
        if not p.is_file(): raise FileNotFoundError(f'필요한 파일이 없습니다: {p}')
    code_dir=Path('/content/gan_bn_diagnostic_code');code_dir.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(code_zip) as z:
        if z.namelist()!=['gan_model.py']: raise ValueError('코드 ZIP 구성이 다릅니다.')
        (code_dir/'gan_model.py').write_bytes(z.read('gan_model.py'))
    sys.path.insert(0,str(code_dir))
    from gan_model import Generator
    def sha(path):
        h=hashlib.sha256()
        with path.open('rb') as f:
            for chunk in iter(lambda:f.read(4*1024*1024),b''): h.update(chunk)
        return h.hexdigest()
    states={name:torch.load(p,map_location='cpu',weights_only=True) for name,p in [('masked',masked),('bed',bed)]}
    # 모든 조합에 완전히 동일한 입력을 사용하며 체크포인트 데이터는 읽기만 한다.
    latent=int(states['masked']['config']['latent_dim'])
    z=torch.randn(16,latent,1,1,generator=torch.Generator(device='cpu').manual_seed(20261101)).to('cuda')
    rows=[]; strips=[]
    for name in ['masked','bed']:
        state=states[name]; config=state['config']
        if int(config['latent_dim'])!=latent: raise ValueError('두 모델의 입력 차원이 다릅니다.')
        for mode in ['eval','train_batch16']:
            # 모드마다 원본 체크포인트에서 새 인스턴스 생성. optimizer/backward/step 없음.
            model=Generator(base=int(config['base']),latent_dim=latent).to('cuda')
            model.load_state_dict(state['generator'],strict=True)
            model.eval() if mode=='eval' else model.train()
            with torch.no_grad(): sample=model(z).detach().cpu()
            if not torch.isfinite(sample).all(): raise RuntimeError(f'출력에 비유한 값: {name} {mode}')
            path=run_root/f'{name}_{mode}.png'
            save_image(sample,path,nrow=4,normalize=True,value_range=(-1,1))
            for i,img in enumerate(sample):
                rows.append({'model':name,'mode':mode,'image_index':i,'minimum':float(img.min()),
                    'maximum':float(img.max()),'mean':float(img.mean()),'std':float(img.std())})
            print(name,mode,'min/max:',float(sample.min()),float(sample.max()),
                  'std:',float(sample.std()),flush=True)
            strips.append((name,mode,path)); del model, sample
    with (run_root/'image_statistics.csv').open('w',newline='',encoding='utf-8-sig') as f:
        w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)
    # 4개 패널을 원본 해상도의 축소본으로 보여준다. 위쪽에 이름을 표시한다.
    tile=512; cap=26
    board=Image.new('RGB',(tile*2,(tile+cap)*2),'white'); draw=ImageDraw.Draw(board)
    for i,(name,mode,path) in enumerate(strips):
        x=(i%2)*tile;y=(i//2)*(tile+cap)
        with Image.open(path) as img: board.paste(ImageOps.contain(img.convert('RGB'),(tile,tile)),(x,y+cap))
        draw.text((x+5,y+5),name+' / '+mode,fill='black')
    board.save(run_root/'batchnorm_four_panel.jpg',quality=94)
    manifest={'status':'completed','gpu':gpu,'seed':20261101,'batch_size':16,'steps':0,
              'weights_updated':False,'checkpoints_saved':False,'checkpoint_sha256':{'masked':sha(masked),'bed':sha(bed)},
              'note':'train_batch16 uses per-batch stats on fresh model; running-stat mutation discarded; not model training'}
    (run_root/'run_manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
    print('완료:',run_root,flush=True)
except BaseException as exc:
    print('진단 오류:',traceback.format_exc(),flush=True)
    if run_root is not None:
        try:
            run_root.mkdir(parents=True,exist_ok=True)
            (run_root/'run_manifest.json').write_text(json.dumps({'status':'failed','error':str(exc),
                'traceback':traceback.format_exc()},ensure_ascii=False,indent=2),encoding='utf-8')
        except BaseException: pass
finally:
    if mounted:
        try: drive.flush_and_unmount()
        except BaseException as exc: print('Drive 동기화 확인 실패:',exc,flush=True)
    print('G4 런타임 반납 요청',flush=True)
    try: runtime.unassign()
    except BaseException as exc: print('반납 확인 실패:',exc,flush=True)
